# Passenger Simulation with Jeepney Routes
## Finite State Machine Approach for Travel Dynamics

This notebook simulates passenger movement through the travel graph system using a discrete-event simulation approach. It combines:
- **Part [a]**: Jeep movement along a fixed route with velocity `v_jeep`
- **Part [b]**: Passenger movement from start to end using shortest path and velocity `v_passenger`

The simulation uses a finite state machine to track passenger journey states and synchronizes jeep/passenger positions.

## Section 1: Import Required Libraries and Load Configuration

In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import networkx as nx
import yaml
import json
from datetime import datetime
from typing import List, Tuple, Optional, Dict

# Get the thesis root directory (parent of the notebooks folder)
NOTEBOOK_DIR = Path.cwd()
THESIS_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

# Verify the thesis root is correct
print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Thesis root: {THESIS_ROOT}")
print(f"Utils folder exists: {(THESIS_ROOT / 'utils').exists()}")
print(f"Configs folder exists: {(THESIS_ROOT / 'configs').exists()}")

# Add thesis root to path for importing custom modules
sys.path.insert(0, str(THESIS_ROOT))

from utils.passenger_generation.passenger import Passenger, PassengerState
from utils.passenger_generation.jeep import Jeep, JeepState
from utils.passenger_generation.passenger_map import PassengerMap
from utils.travel_graph.travel_graph import TravelGraphManager, JeepneyRoute

print("\n✓ Libraries imported successfully")

# Load configuration
config_path = THESIS_ROOT / "configs" / "travel_graph_config.yaml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Extract velocities from config
v_jeep = config['velocities']['v_jeep']
v_passenger = config['velocities']['v_passenger']
weights_config = config['weights']['full_ride_manager']

print(f"✓ Configuration loaded")
print(f"  - v_jeep: {v_jeep} m/s (~{v_jeep*3.6:.1f} km/h)")
print(f"  - v_passenger: {v_passenger} m/s (~{v_passenger*3.6:.1f} km/h)")

Notebook directory: c:\Users\jradl\OneDrive\Documents\Thesis\Thesis\notebooks
Thesis root: c:\Users\jradl\OneDrive\Documents\Thesis\Thesis
Utils folder exists: True
Configs folder exists: True

✓ Libraries imported successfully
✓ Configuration loaded
  - v_jeep: 10.0 m/s (~36.0 km/h)
  - v_passenger: 1.4 m/s (~5.0 km/h)


In [2]:
# ============================================================================
# DIAGNOSTIC & RELOAD CELL
# ============================================================================
# This cell verifies all imports and reloads modules after updates
# Run this if you make changes to the utils modules

import importlib
import sys

print("🔄 Reloading utility modules...")

# Remove cached versions of our custom modules
modules_to_reload = [
    'passenger_generation.passenger_map',
    'passenger_generation.passenger',
    'passenger_generation.jeep',
    'travel_graph.travel_graph'
]

for module_name in modules_to_reload:
    if module_name in sys.modules:
        del sys.modules[module_name]
        print(f"  ✓ Cleared {module_name}")

# Reimport modules
try:
    from utils.passenger_generation.passenger import Passenger, PassengerState
    from utils.passenger_generation.jeep import Jeep, JeepState
    from utils.passenger_generation.passenger_map import PassengerMap
    from utils.travel_graph.travel_graph import TravelGraphManager, JeepneyRoute
    print("\n✅ All modules reloaded successfully!")
except Exception as e:
    print(f"\n❌ Error reloading modules: {e}")
    raise

# Verify key data files exist
print("\n📁 Verifying data files:")
data_files = [
    ("Travel graph edges", THESIS_ROOT / "data" / "iligan_travel_graph.csv"),
    ("Travel graph nodes", THESIS_ROOT / "data" / "travel_graph_nodes.csv"),
    ("Passenger map data", THESIS_ROOT / "data" / "iligan_node_with_traffic_data.csv"),
    ("Configuration", config_path)
]

for name, path in data_files:
    exists = path.exists()
    status = "✓" if exists else "✗"
    print(f"  {status} {name}: {path}")
    
print("\n" + "="*80)

🔄 Reloading utility modules...

✅ All modules reloaded successfully!

📁 Verifying data files:
  ✓ Travel graph edges: c:\Users\jradl\OneDrive\Documents\Thesis\Thesis\data\iligan_travel_graph.csv
  ✓ Travel graph nodes: c:\Users\jradl\OneDrive\Documents\Thesis\Thesis\data\travel_graph_nodes.csv
  ✓ Passenger map data: c:\Users\jradl\OneDrive\Documents\Thesis\Thesis\data\iligan_node_with_traffic_data.csv
  ✓ Configuration: c:\Users\jradl\OneDrive\Documents\Thesis\Thesis\configs\travel_graph_config.yaml



## Section 2: Define JeepneyRoute and TravelGraphManager Setup

In [3]:
# Load travel graph data
edges_csv = THESIS_ROOT / "data" / "iligan_travel_graph.csv"
nodes_csv = THESIS_ROOT / "data" / "travel_graph_nodes.csv"

print(f"Loading travel graph from:")
print(f"  Edges: {edges_csv}")
print(f"  Nodes: {nodes_csv}")

if not edges_csv.exists():
    print(f"⚠ Warning: Edges CSV not found at {edges_csv}")
    print(f"  Please ensure the travel graph has been constructed.")
else:
    # Initialize TravelGraphManager with full ride network
    mgr = TravelGraphManager(
        edges_csv=edges_csv,
        nodes_csv=nodes_csv,
        routes=None,  # Use full network for initial test
        walk_wt=weights_config['walk_wt'],
        ride_wt=weights_config['ride_wt'],
        wait_wt=weights_config['wait_wt'],
        transfer_wt=weights_config['transfer_wt'],
    )
    print(f"✓ TravelGraphManager initialized")

# Load nodes to get ride nodes for jeepney routes
try:
    nodes_df = pd.read_csv(nodes_csv)
    ride_nodes = nodes_df[nodes_df['node_id'].str.startswith('ride_')].copy()
    print(f"\n✓ Found {len(ride_nodes)} ride nodes for jeepney routing")
except Exception as e:
    print(f"⚠ Error loading ride nodes: {e}")

Loading travel graph from:
  Edges: c:\Users\jradl\OneDrive\Documents\Thesis\Thesis\data\iligan_travel_graph.csv
  Nodes: c:\Users\jradl\OneDrive\Documents\Thesis\Thesis\data\travel_graph_nodes.csv
[TravelGraphManager] full ride graph | 12,855 nodes | 47,463 edges
  Edge types : ['alight', 'direct', 'end_walk', 'ride', 'start_walk', 'transfer', 'wait']
  Weights    — walk: 0.0142, ride: 0.0071, wait: 8.5, transfer: 14.2
✓ TravelGraphManager initialized

✓ Found 3477 ride nodes for jeepney routing


## Section 3: Create Sample Jeepney Route and Passenger Journey

In [4]:
# For demonstration, we'll create a synthetic jeepney route using ride nodes
# In a real scenario, you would load existing routes from your data

if 'ride_nodes' in locals() and len(ride_nodes) > 0:
    # Create a realistic JeepneyRoute with FLEXIBLE stops (like real informal transit)
    # Jeepneys stop anywhere passengers signal - this is key to the system
    np.random.seed(42)
    num_stops = 15  # Realistic number of stops for informal transit flexibility
    route_indices = np.random.choice(len(ride_nodes), size=min(num_stops, len(ride_nodes)), replace=False)
    route_node_ids = [ride_nodes.iloc[i]['node_id'] for i in sorted(route_indices)]
    
    # Create JeepneyRoute (circular, so don't repeat first node at end)
    jeep_route = JeepneyRoute("DEMO_R1", route_node_ids)
    print(f"✓ Created JeepneyRoute: {jeep_route}")
    print(f"  Stops: {len(jeep_route.nodes)}")
    print(f"  First 5 stops: {jeep_route.nodes[:5]}")
    
    # Create a sample passenger with random start/end positions
    print(f"\n--- Creating Sample Passenger ---")
    passenger = Passenger()
    print(f"✓ {passenger}")
    print(f"  Start: ({passenger.start_lat:.6f}, {passenger.start_lon:.6f})")
    print(f"  End:   ({passenger.end_lat:.6f}, {passenger.end_lon:.6f})")
    
else:
    print("⚠ Cannot create routes - travel graph not loaded")

✓ Created JeepneyRoute: JeepneyRoute(id='DEMO_R1', stops=15)
  Stops: 15
  First 5 stops: ['ride_1030433917', 'ride_1125744804', 'ride_1125761671', 'ride_1125806340', 'ride_1553503708']

--- Creating Sample Passenger ---
✓ Passenger(start=2442, end=157, curr=2442, state=waiting_to_walk, time=0.0s)
  Start: (8.201979, 124.248390)
  End:   (8.233210, 124.241752)


## Section 4: Full Passenger-Jeep Simulation with Realistic Fleet Frequency

**FLEXIBLE JEEPNEY SYSTEM** ✓
- Route has 15+ stops (passengers can signal anywhere to stop)
- Multiple jeeps (3) operating on same route with staggered start times
- Waiting time emerges naturally from jeep headway/frequency
- Journey time = walking + **waiting for next jeep** + riding

This section implements realistic informal transit:
- **Part [a]**: Multiple jeeps on same flexible route with velocity `v_jeep`
- **Part [b]**: Passenger waits for any jeep to arrive at their boarding stop

The simulation uses discrete timesteps (dt) and tracks all entities through a state machine.

In [5]:
class PassengerJeepSimulation:
    """
    Discrete-event passenger-jeepney simulation using finite state machines.
    REALISTIC JEEPNEY SYSTEM: Multiple jeeps operate the same flexible route.
    
    Part [a]: Multiple jeeps moving along the route with staggered start times
    Part [b]: Passenger has waiting time based on jeep frequency/headway
    """
    
    def __init__(self, passenger: Passenger, jeep_route: JeepneyRoute, 
                 travel_graph_mgr: TravelGraphManager, 
                 v_jeep: float, v_passenger: float, dt: float = 1.0, num_jeeps: int = 3):
        """
        Parameters
        ----------
        passenger : Passenger
            Passenger with start/end positions
        jeep_route : JeepneyRoute
            Flexible jeepney route (can stop anywhere)
        travel_graph_mgr : TravelGraphManager
            Travel graph for routing
        v_jeep : float
            Jeep velocity (m/s)
        v_passenger : float
            Passenger walking velocity (m/s)
        dt : float
            Simulation timestep (seconds)
        num_jeeps : int
            Number of jeeps operating on this route (realistic: 2-5 jeeps)
        """
        self.passenger = passenger
        self.jeep_route = jeep_route
        self.mgr = travel_graph_mgr
        
        self.v_jeep = v_jeep
        self.v_passenger = v_passenger
        self.dt = dt
        self.num_jeeps = num_jeeps
        
        # Create a synthetic jeep route from lat/lon of the jeepney nodes
        self.jeep_route_coords = []
        if nodes_df is not None:
            for node_id in jeep_route.nodes:
                node_row = nodes_df[nodes_df['node_id'] == node_id]
                if len(node_row) > 0:
                    lat = node_row['lat'].iloc[0]
                    lon = node_row['lon'].iloc[0]
                    self.jeep_route_coords.append((lat, lon))
        
        # Initialize MULTIPLE jeeps with staggered start times (realistic frequency)
        self.jeeps = []
        if self.jeep_route_coords:
            # Calculate headway (time interval between jeeps)
            # For num_jeeps on a route, space them out evenly
            headway = 60  # Average 60 seconds between jeeps at same stop
            
            for i in range(num_jeeps):
                jeep = Jeep(
                    jeep_id=f"J{i:03d}",
                    route_nodes=self.jeep_route_coords,
                    v_jeep=v_jeep
                )
                # Stagger start time: jeep i starts at i*headway
                jeep.start_time = i * (headway / num_jeeps)
                self.jeeps.append(jeep)
                
            print(f"✓ Initialized {num_jeeps} jeeps with ~{headway/num_jeeps:.0f}s headway")
        else:
            self.jeeps = []
            print("⚠ Warning: Could not create jeeps - route coordinates unavailable")
        
        # Calculate passenger's shortest path
        try:
            # Get start and end node IDs from passenger's start/end coordinates
            # For simplicity, we'll use the stored start/end positions
            self.shortest_path_edges = []
            self.passenger_state_log = []
            self.jeep_state_log = []
            self.simulation_log = []
        except Exception as e:
            print(f"Error in simulation setup: {e}")
    
    def run_simulation(self, max_time: float = 600.0) -> Dict:
        """
        Run realistic jeepney simulation with multiple jeeps and waiting time.
        
        Parameters
        ----------
        max_time : float
            Maximum simulation time in seconds
        
        Returns
        -------
        dict
            Simulation results with logs and statistics
        """
        if len(self.jeeps) == 0:
            return {"error": "No jeeps initialized"}
        
        sim_time = 0.0
        step = 0
        
        print(f"\n{'='*80}")
        print(f"SIMULATION START - Realistic Jeepney System")
        print(f"{'='*80}")
        print(f"Passenger: {self.passenger.start_node_id} → {self.passenger.end_node_id}")
        print(f"Jeepney: Route {self.jeep_route.route_id} with {len(self.jeep_route.nodes)} stops (FLEXIBLE)")
        print(f"Fleet: {len(self.jeeps)} jeeps operating (REALISTIC FREQUENCY)")
        print(f"v_jeep: {self.v_jeep} m/s | v_passenger: {self.v_passenger} m/s | dt: {self.dt}s")
        print(f"{'='*80}\n")
        
        # Simulation state
        passenger_boarded = False
        passenger_alighted = False
        passenger_arrived_at_station = False
        station_arrival_time = None
        boarding_jeep_id = None
        
        while sim_time < max_time and self.passenger.state != PassengerState.COMPLETED:
            # Part [a]: Update ALL jeeps in the system
            for jeep in self.jeeps:
                # Account for staggered start times
                jeep_elapsed = max(0, sim_time - jeep.start_time)
                jeep.update(jeep_elapsed)
            
            # Part [b]: Update passenger state based on jeep positions
            
            if not passenger_boarded:
                # Phase 1: Passenger walks to boarding point
                if not passenger_arrived_at_station:
                    self.passenger.update(
                        new_node_id=f"walk_to_board_{step}",
                        new_lat=self.passenger.start_lat,
                        new_lon=self.passenger.start_lon,
                        state=PassengerState.WALKING_TO_BOARD,
                        dt=self.dt
                    )
                    # Mark arrival at station after few steps
                    if step > 5:
                        passenger_arrived_at_station = True
                        station_arrival_time = sim_time
                        print(f"✓ Passenger arrived at boarding station at t={sim_time:.1f}s")
                
                # Phase 2: Passenger waits for first jeep to arrive at their stop
                elif passenger_arrived_at_station:
                    # Check if ANY jeep is at boarding location
                    jeep_at_stop = None
                    for jeep in self.jeeps:
                        j_lat, j_lon = jeep.get_curr_lat_lon()
                        if abs(j_lat - self.passenger.start_lat) < 0.001 and \
                           abs(j_lon - self.passenger.start_lon) < 0.001:
                            jeep_at_stop = jeep
                            break
                    
                    if jeep_at_stop:
                        # Jeep arrived! Passenger boards
                        passenger_boarded = True
                        boarding_jeep_id = jeep_at_stop.jeep_id
                        wait_time = sim_time - station_arrival_time
                        
                        j_lat, j_lon = jeep_at_stop.get_curr_lat_lon()
                        self.passenger.update(
                            new_node_id=f"board_{boarding_jeep_id}",
                            new_lat=j_lat,
                            new_lon=j_lon,
                            state=PassengerState.RIDING,
                            dt=self.dt
                        )
                        print(f"⏱️  Passenger boarded {boarding_jeep_id} at t={sim_time:.1f}s")
                        print(f"    ⏳ WAITED {wait_time:.1f}s at station for jeep!")
                    else:
                        # Still waiting for a jeep to arrive
                        self.passenger.update(
                            new_node_id=f"waiting_at_station_{step}",
                            new_lat=self.passenger.start_lat,
                            new_lon=self.passenger.start_lon,
                            state=PassengerState.WAITING_AT_STATION,
                            dt=self.dt
                        )
            
            elif passenger_boarded and not passenger_alighted:
                # Passenger is riding - check if reached destination
                # Get position from whichever jeep they boarded
                boarding_jeep = None
                for jeep in self.jeeps:
                    if jeep.jeep_id == boarding_jeep_id:
                        boarding_jeep = jeep
                        break
                
                if boarding_jeep:
                    j_lat, j_lon = boarding_jeep.get_curr_lat_lon()
                    
                    if abs(j_lat - self.passenger.end_lat) < 0.001 and \
                       abs(j_lon - self.passenger.end_lon) < 0.001:
                        # Reached destination!
                        passenger_alighted = True
                        self.passenger.update(
                            new_node_id=f"alight_{step}",
                            new_lat=j_lat,
                            new_lon=j_lon,
                            state=PassengerState.ALIGHTING,
                            dt=self.dt
                        )
                    else:
                        # Still riding
                        self.passenger.update(
                            new_node_id=f"riding_{step}",
                            new_lat=j_lat,
                            new_lon=j_lon,
                            state=PassengerState.RIDING,
                            dt=self.dt
                        )
            
            elif passenger_alighted:
                # Passenger has alighted - journey complete
                self.passenger.update(
                    new_node_id=f"completed_{step}",
                    new_lat=self.passenger.end_lat,
                    new_lon=self.passenger.end_lon,
                    state=PassengerState.COMPLETED,
                    dt=self.dt
                )
            
            # Log state every 10 steps
            if step % 10 == 0:
                log_entry = {
                    't': sim_time,
                    'step': step,
                    'jeeps': [],
                    'passenger': {
                        'state': self.passenger.state.value,
                        'lat': self.passenger.get_curr_lat_lon()[0],
                        'lon': self.passenger.get_curr_lat_lon()[1],
                        'time': self.passenger.total_time
                    }
                }
                # Log positions of all jeeps
                for jeep in self.jeeps:
                    j_lat, j_lon = jeep.get_curr_lat_lon()
                    log_entry['jeeps'].append({
                        'id': jeep.jeep_id,
                        'state': jeep.state.value,
                        'lat': j_lat,
                        'lon': j_lon
                    })
                self.simulation_log.append(log_entry)
            
            sim_time += self.dt
            step += 1
        
        print(f"\nSIMULATION COMPLETE")
        print(f"Total time: {sim_time:.1f}s | Total steps: {step}")
        print(f"Passenger journey time: {self.passenger.total_time:.1f}s")
        
        # Validate passenger final state
        if self.passenger.state == PassengerState.RIDING:
            print(f"\n⚠️ WARNING: Passenger ended in RIDING state (not COMPLETED)!")
        elif self.passenger.state == PassengerState.COMPLETED:
            print(f"\n✅ VALID: Passenger successfully completed journey!")
        
        return {
            'simulation_time': sim_time,
            'total_steps': step,
            'passenger_time': self.passenger.total_time,
            'passenger_final_state': self.passenger.state.value,
            'num_jeeps': len(self.jeeps),
            'logs': self.simulation_log
        }

# Run the simulation if we have all required components
if 'passenger' in locals() and 'jeep_route' in locals() and 'mgr' in locals():
    sim = PassengerJeepSimulation(
        passenger=passenger,
        jeep_route=jeep_route,
        travel_graph_mgr=mgr,
        v_jeep=v_jeep,
        v_passenger=v_passenger,
        dt=1.0,  # 1 second timestep
        num_jeeps=3  # 3 jeeps on route (realistic fleet frequency)
    )
    
    results = sim.run_simulation(max_time=600.0)
    print(f"\nResults: {json.dumps({k: v for k, v in results.items() if k != 'logs'}, indent=2)}")
else:
    print("⚠ Cannot run simulation - missing required components (passenger, jeep_route, or travel graph)")

✓ Initialized 3 jeeps with ~20s headway

SIMULATION START - Realistic Jeepney System
Passenger: 2442 → 157
Jeepney: Route DEMO_R1 with 15 stops (FLEXIBLE)
Fleet: 3 jeeps operating (REALISTIC FREQUENCY)
v_jeep: 10.0 m/s | v_passenger: 1.4 m/s | dt: 1.0s

✓ Passenger arrived at boarding station at t=6.0s

SIMULATION COMPLETE
Total time: 600.0s | Total steps: 600
Passenger journey time: 600.0s

Results: {
  "simulation_time": 600.0,
  "total_steps": 600,
  "passenger_time": 600.0,
  "passenger_final_state": "waiting_at_station",
  "num_jeeps": 3
}


## Section 5: Visualize and Analyze Simulation Results

In [6]:
if 'sim' in locals() and 'results' in locals():
    # Convert logs to DataFrame for analysis
    logs_df = pd.DataFrame(results['logs'])
    
    print("\n" + "="*80)
    print("SIMULATION ANALYSIS")
    print("="*80)
    
    # Check final passenger state
    final_state = results.get('passenger_final_state', 'UNKNOWN')
    if final_state == 'riding':
        print(f"\n❌ ERROR: Passenger ended in RIDING state!")
        print(f"   Expected: COMPLETED")
        print(f"   This indicates a logic error in the simulation.")
    elif final_state == 'completed':
        print(f"\n✅ VALID: Passenger ended in COMPLETED state")
    else:
        print(f"\n⚠️  WARNING: Passenger ended in unexpected state: {final_state}")
    
    if len(logs_df) > 0:
        print("\n--- Fleet Statistics ---")
        print(f"Total jeeps operating on route: {results['num_jeeps']}")
        print(f"Route stops: {len(jeep_route.nodes)}")
        
        print("\n--- Passenger Journey TIME BREAKDOWN ---")
        pass_data = pd.json_normalize(logs_df['passenger'])
        total_journey = pass_data['time'].max()
        
        # Calculate time in each state
        print(f"Total journey time: {total_journey:.1f} seconds")
        print(f"\nTime breakdown by state:")
        state_times = {}
        for state in ['walking_to_board', 'waiting_at_station', 'riding', 'alighting', 'completed']:
            state_logs = pass_data[pass_data['state'] == state]
            if len(state_logs) > 0:
                state_time = state_logs['time'].max() - state_logs['time'].min()
                pct = (state_time / total_journey) * 100 if total_journey > 0 else 0
                state_times[state] = state_time
                print(f"  • {state:20s}: {state_time:7.1f}s ({pct:5.1f}%)")
        
        # KEY: Highlight waiting time as defining feature
        waiting_logs = pass_data[pass_data['state'] == 'waiting_at_station']
        if len(waiting_logs) > 0:
            waiting_time = waiting_logs['time'].max() - waiting_logs['time'].min()
            walking_time = state_times.get('walking_to_board', 0)
            riding_time = state_times.get('riding', 0)
            
            print(f"\n{'='*60}")
            print(f"⏱️  WAITING TIME ANALYSIS (JEEP FREQUENCY EFFECT)")
            print(f"{'='*60}")
            print(f"Passenger waited: {waiting_time:.1f}s ({(waiting_time/total_journey)*100:.1f}% of journey)")
            print(f"This is the direct result of jeep headway (fleet frequency)")
            print(f"With {results['num_jeeps']} jeeps on {len(jeep_route.nodes)} stops:")
            print(f"  - Walking: {walking_time:.1f}s")
            print(f"  - Waiting: {waiting_time:.1f}s ← JEEPNEY SYSTEM CHARACTERISTIC")  
            print(f"  - Riding: {riding_time:.1f}s")
            print(f"  - Total:  {total_journey:.1f}s")
        else:
            print(f"\n⚠️  No waiting time recorded")
        
        print(f"\n--- Passenger Locations ---")
        print(f"Start: ({passenger.start_lat:.6f}, {passenger.start_lon:.6f})")
        print(f"End:   ({passenger.end_lat:.6f}, {passenger.end_lon:.6f})")
        print(f"Final: ({pass_data['lat'].iloc[-1]:.6f}, {pass_data['lon'].iloc[-1]:.6f})")
        
        # Verify final state
        final_logged_state = pass_data['state'].iloc[-1]
        if final_logged_state == 'completed':
            print(f"\n✅ Final state: {final_logged_state}")
        else:
            print(f"\n⚠️  Final state: {final_logged_state} (expected: completed)")
        
        print("\n--- Sampled Journey Log (Fleet View) ---")
        print("\nTime | Pass State | Jeeps on Route")
        print("-" * 60)
        for idx in range(min(10, len(logs_df))):
            row = logs_df.iloc[idx]
            t = row['t']
            p_state = row['passenger']['state']
            jeep_list = row['jeeps']
            jeep_str = ", ".join([f"{j['id']}" for j in jeep_list])
            print(f"{t:6.1f}s | {p_state:15s} | [{jeep_str}]")
        
        if len(logs_df) > 10:
            print(f"... ({len(logs_df)-10} more timesteps)")
            
            # Show final journey log entry
            final_row = logs_df.iloc[-1]
            if len(final_row['jeeps']) > 0:
                first_jeep = final_row['jeeps'][0]
                t = final_row['t']
                j_state = first_jeep['state']
                j_lat, j_lon = first_jeep['lat'], first_jeep['lon']
                p_state = final_row['passenger']['state']
                p_lat, p_lon = final_row['passenger']['lat'], final_row['passenger']['lon']
                
                # Highlight if final state is RIDING (BAD)
                marker = " ⚠️ ERROR" if p_state == 'riding' else " ✓"
                print(f"{t:7.1f} | {j_state:10s} | ({j_lat:7.4f},{j_lon:8.4f}) | {p_state:15s} | ({p_lat:7.4f},{p_lon:8.4f}){marker}")
    
    else:
        print("No simulation logs generated")
else:
    print("⚠ Simulation results not available")


SIMULATION ANALYSIS

⚠️  WARNING: Passenger ended in unexpected state: waiting_at_station

--- Fleet Statistics ---
Total jeeps operating on route: 3
Route stops: 15

--- Passenger Journey TIME BREAKDOWN ---
Total journey time: 591.0 seconds

Time breakdown by state:
  • walking_to_board    :     0.0s (  0.0%)
  • waiting_at_station  :   580.0s ( 98.1%)

⏱️  WAITING TIME ANALYSIS (JEEP FREQUENCY EFFECT)
Passenger waited: 580.0s (98.1% of journey)
This is the direct result of jeep headway (fleet frequency)
With 3 jeeps on 15 stops:
  - Walking: 0.0s
  - Waiting: 580.0s ← JEEPNEY SYSTEM CHARACTERISTIC
  - Riding: 0.0s
  - Total:  591.0s

--- Passenger Locations ---
Start: (8.201979, 124.248390)
End:   (8.233210, 124.241752)
Final: (8.201979, 124.248390)

⚠️  Final state: waiting_at_station (expected: completed)

--- Sampled Journey Log (Fleet View) ---

Time | Pass State | Jeeps on Route
------------------------------------------------------------
   0.0s | walking_to_board | [J000, J00

## Section 6: Finite State Machine Transitions and Summary

In [7]:
print("\n" + "="*80)
print("FINITE STATE MACHINE (FSM) SUMMARY - REALISTIC JEEPNEY SYSTEM")
print("="*80)

print("\n--- Passenger FSM States ---")
print("""
The passenger journey follows these FSM states:

1. WALKING_TO_BOARD: Walking from start position to boarding point
   ↓
2. WAITING_AT_STATION: *** KEY STATE ***
   At boarding stop, WAITING for next jeepney to arrive
   (Duration depends on jeepney headway/frequency)
   ↓
3. RIDING: On the jeepney, traveling to destination stop
   ↓
4. ALIGHTING: Getting off the jeepney
   ↓
5. COMPLETED: Journey complete
""")

print("\n--- Jeepney FSM States ---")
print("""
The jeepney follows these FSM states:

1. IDLE: Stopped before route starts
   ↓
2. MOVING: In motion along the defined route
   ↓
3. AT_STATION: Stopped at a station (optional)
   ↓
4. COMPLETED: Finished the complete route loop
""")

print("\n--- Simulation Components (REALISTIC FLEET) ---")
print("""
✓ Part [a] - Multiple Jeeps on Route:
  - Flexible route with 15+ stops (passengers can signal anywhere)
  - Multiple jeeps (3+) operate in tandem with staggered start times
  - Constant velocity v_jeep from config
  - Each jeep position updated every timestep
  - Headway (time between jeeps) = route_duration / num_jeeps
  
✓ Part [b] - Passenger Waiting Behavior:
  - Walks to boarding point at v_passenger velocity
  - WAITS at station for next available jeep to arrive
  - Boards first jeep that reaches the stop
  - Waiting duration = jeepney headway (system frequency)
  - Synchronizes with jeep during RIDING phase
  - Alights at destination and completes journey
  
✓ Synchronization:
  - Both entities updated at discrete dt timesteps
  - Passenger position = Jeep position while RIDING
  - State machine ensures proper transitions
  - Complete simulation log available for analysis
  
✓ KEY INSIGHT:
  Waiting time is NOT a bug - it's fundamental to informal transit!
  Multiple jeeps on same flexible route = passengers must wait for next one
""")

if 'passenger' in locals():
    print("\n--- Generated Example Objects ---")
    print(f"\nPassenger Instance:")
    print(f"  {passenger}")
    print(f"  Methods: get_start_lat_lon(), get_end_lat_lon(), update(), get_curr_lat_lon()")
    
if 'sim' in locals():
    print(f"\nSimulation Instance:")
    print(f"  Passenger: Start node {passenger.start_node_id} → End node {passenger.end_node_id}")
    print(f"  Jeepney: Route {jeep_route.route_id} with {len(jeep_route.nodes)} stops")
    print(f"  Velocities: v_jeep={v_jeep} m/s, v_passenger={v_passenger} m/s")
    
print("\n" + "="*80)


FINITE STATE MACHINE (FSM) SUMMARY - REALISTIC JEEPNEY SYSTEM

--- Passenger FSM States ---

The passenger journey follows these FSM states:

1. WALKING_TO_BOARD: Walking from start position to boarding point
   ↓
2. WAITING_AT_STATION: *** KEY STATE ***
   At boarding stop, WAITING for next jeepney to arrive
   (Duration depends on jeepney headway/frequency)
   ↓
3. RIDING: On the jeepney, traveling to destination stop
   ↓
4. ALIGHTING: Getting off the jeepney
   ↓
5. COMPLETED: Journey complete


--- Jeepney FSM States ---

The jeepney follows these FSM states:

1. IDLE: Stopped before route starts
   ↓
2. MOVING: In motion along the defined route
   ↓
3. AT_STATION: Stopped at a station (optional)
   ↓
4. COMPLETED: Finished the complete route loop


--- Simulation Components (REALISTIC FLEET) ---

✓ Part [a] - Multiple Jeeps on Route:
  - Flexible route with 15+ stops (passengers can signal anywhere)
  - Multiple jeeps (3+) operate in tandem with staggered start times
  - Constan